# ⚖️ Fetal Birth Weight Estimation — Model Training
**MedPredict | JSS STU | Dept. of CS&E | Academic Year 2025-26**

---
### Pipeline
1. Load / generate fetal biometry dataset
2. EDA & cleaning
3. Feature engineering → 32 features (matches app.py)
4. Train/test split
5. Model comparison — **Regression task** (XGBoost, LightGBM, Random Forest, SVR, Ridge)
6. Optuna tuning
7. Evaluation: RMSE, MAE, R², ±10% accuracy (clinically standard)
8. Save model + scaler
9. Integration snippet for `app.py`

**Task Type:** Regression — predict birth weight in grams  
**Hadlock Formula:** Used as baseline; ML model aims to improve over it  
**Dataset:** INTERGROWTH-21st / Synthetic biometry data matching app.py inputs

## 0. Install Dependencies

In [ ]:
!pip install lightgbm xgboost optuna scikit-learn pandas numpy matplotlib seaborn shap --quiet

## 1. Imports & Config

In [ ]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.svm import SVR

import lightgbm as lgb
import xgboost as xgb
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42
os.makedirs('model', exist_ok=True)
print('✅ All imports successful')

## 2. Load Dataset

**Option A** — Real dataset: Upload a CSV with ultrasound biometry + actual birth weights  
**Option B** — Synthetic generator calibrated to INTERGROWTH-21st growth standards (recommended for prototyping)

Required columns: `GestationalAge, MaternalAge, MaternalWeight, MaternalHeight, PrePregnancyBMI,
GestationalWeightGain, FundalHeight, BPD, HC, AC, FL, Parity, Gravida, MaternalGlucose, AFI, ActualBirthWeight_g`

In [ ]:
def generate_fetal_bw_dataset(n=4000, seed=42):
    """
    Synthetic fetal biometry dataset calibrated to INTERGROWTH-21st standards.
    BPD, HC, AC, FL are correlated with GA following real ultrasound growth curves.
    """
    rng = np.random.default_rng(seed)

    ga   = rng.uniform(28, 42, n)   # gestational age in weeks

    # ── INTERGROWTH-21st approximations ─────────────────────────────
    # BPD (cm): ~0.27*GA - 0.2 ± noise
    bpd = (0.27 * ga - 0.25 + rng.normal(0, 0.25, n)).clip(3, 12)
    # HC (cm): ~0.85*GA + 2.8 ± noise  (Hadlock et al.)
    hc  = (0.85 * ga + 2.8  + rng.normal(0, 0.8,  n)).clip(10, 38)
    # AC (cm): ~0.78*GA + 2.2 ± noise
    ac  = (0.78 * ga + 2.0  + rng.normal(0, 1.2,  n)).clip(8, 42)
    # FL (cm): ~0.16*GA - 0.5 ± noise
    fl  = (0.16 * ga - 0.5  + rng.normal(0, 0.3,  n)).clip(2, 9.5)

    m_age   = rng.normal(28, 6, n).clip(15, 50).astype(int)
    m_wt    = rng.normal(68, 12, n).clip(38, 130)
    m_ht    = rng.normal(161, 7, n).clip(135, 190)
    pp_bmi  = rng.normal(22, 4, n).clip(14, 45)
    wg      = rng.normal(12, 4, n).clip(0, 35)    # gestational weight gain kg
    fh      = (0.95 * ga + rng.normal(0, 1.5, n)).clip(15, 46)  # fundal height
    glucose = rng.normal(90, 20, n).clip(60, 250)
    afi     = rng.normal(13, 4, n).clip(1, 35)    # amniotic fluid index
    parity  = rng.choice([0,1,2,3,4], n, p=[0.38,0.32,0.18,0.08,0.04])
    gravida = parity + rng.choice([0,1,2], n, p=[0.60,0.30,0.10])

    # ── Hadlock Formula 4 as base truth + noise ──────────────────────
    log_efw = (
        1.3596 + 0.0064 * hc + 0.0424 * ac +
        0.174  * fl + 0.00061 * bpd * ac - 0.00386 * ac * fl
    )
    efw_base = 10 ** log_efw

    # Maternal adjustments (same as app.py)
    efw = efw_base.copy()
    efw[glucose > 130]  *= rng.uniform(1.02, 1.06, n)[glucose > 130]
    efw[wg > 20]        *= rng.uniform(1.01, 1.03, n)[wg > 20]
    efw[parity > 3]     *= rng.uniform(1.01, 1.02, n)[parity > 3]

    # Add measurement uncertainty ±8%
    actual_bw = (efw * rng.uniform(0.93, 1.10, n)).clip(500, 5500)

    df = pd.DataFrame({
        'GestationalAge': ga.round(1),
        'MaternalAge': m_age,
        'MaternalWeight': m_wt.round(1),
        'MaternalHeight': m_ht.round(0),
        'PrePregnancyBMI': pp_bmi.round(1),
        'GestationalWeightGain': wg.round(1),
        'FundalHeight': fh.round(1),
        'BPD': bpd.round(2),
        'HC': hc.round(2),
        'AC': ac.round(2),
        'FL': fl.round(2),
        'Parity': parity,
        'Gravida': gravida.clip(1, 12).astype(int),
        'MaternalGlucose': glucose.round(0),
        'AFI': afi.round(1),
        'ActualBirthWeight_g': actual_bw.round(0)
    })
    return df


# ── Load your real dataset if available ──────────────────────────
# df = pd.read_csv('fetal_biometry.csv')

df = generate_fetal_bw_dataset(4000, SEED)
print(f'Dataset shape: {df.shape}')
print(f'Birth weight range: {df["ActualBirthWeight_g"].min():.0f}g – {df["ActualBirthWeight_g"].max():.0f}g')
print(f'Mean birth weight: {df["ActualBirthWeight_g"].mean():.0f}g')
df.head()

## 3. EDA

In [ ]:
print(df.describe().T.to_string())

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
features_to_plot = ['GestationalAge','HC','AC','FL','BPD',
                     'MaternalWeight','MaternalGlucose','AFI']
for i, col in enumerate(features_to_plot):
    axes[i].scatter(df[col], df['ActualBirthWeight_g'],
                    alpha=0.3, s=5, color='#EC4899')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Birth Weight (g)')
    axes[i].set_title(f'{col} vs Birth Weight')

plt.suptitle('Feature vs Birth Weight Scatter Plots', fontsize=13, color='#9D174D')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target
corr = df.corr()['ActualBirthWeight_g'].sort_values(ascending=False)
print('=== Correlation with Birth Weight ===')
print(corr.to_string())

## 4. Feature Engineering (32 Features)

In [ ]:
def engineer_fbw_features_df(df):
    """Matches engineer_fbw_features() in app.py exactly."""
    d = df.copy()

    # === Raw inputs (15) ===
    # GestationalAge, MaternalAge, MaternalWeight, MaternalHeight,
    # PrePregnancyBMI, GestationalWeightGain, FundalHeight,
    # BPD, HC, AC, FL, Parity, Gravida, MaternalGlucose, AFI

    # === Ultrasound-derived ratios (5) ===
    d['HC_AC_Ratio']           = d['HC'] / (d['AC'] + 1e-6)
    d['FL_HC_Ratio']           = d['FL'] / (d['HC'] + 1e-6)
    d['FL_AC_Ratio']           = d['FL'] / (d['AC'] + 1e-6)
    d['BPD_AC_Product']        = d['BPD'] * d['AC']
    d['HadlockScore']          = (0.5336 * d['AC'] +
                                   0.1714 * d['FL'] +
                                   0.0664 * d['HC']) / 10

    # === Maternal features (4) ===
    d['CurrentBMI']            = d['MaternalWeight'] / ((d['MaternalHeight'] / 100) ** 2 + 1e-6)
    d['BMIChange']             = d['CurrentBMI'] - d['PrePregnancyBMI']
    d['WeightGainPerWeek']     = d['GestationalWeightGain'] / (d['GestationalAge'] + 1e-6)
    d['FH_GA_Ratio']           = d['FundalHeight'] / (d['GestationalAge'] + 1e-6)

    # === Clinical flags (8) ===
    d['ExcessWeightGain']      = (d['GestationalWeightGain'] > 16).astype(int)
    d['LowWeightGain']         = (d['GestationalWeightGain'] < 7).astype(int)
    d['ObeseMother']           = (d['PrePregnancyBMI'] >= 30).astype(int)
    d['HighGlucose']           = (d['MaternalGlucose'] > 130).astype(int)
    d['Macrosomia_Risk']       = ((d['AC'] > 35) & (d['GestationalAge'] > 36)).astype(int)
    d['IUGR_Risk']             = ((d['AC'] < 28) & (d['GestationalAge'] > 36)).astype(int)
    d['Polyhydramnios']        = (d['AFI'] > 24).astype(int)
    d['Oligohydramnios']       = (d['AFI'] < 5).astype(int)

    # === GA category (ordinal) ===
    d['TermCategory']          = pd.cut(
        d['GestationalAge'],
        bins=[0, 33.9, 36.9, 41.0, 100],
        labels=[0, 1, 2, 3]
    ).astype(int)

    return d


df_feat = engineer_fbw_features_df(df)

FEATURE_COLS = [
    # Raw
    'GestationalAge', 'MaternalAge', 'MaternalWeight', 'MaternalHeight',
    'PrePregnancyBMI', 'GestationalWeightGain', 'FundalHeight',
    'BPD', 'HC', 'AC', 'FL',
    'Parity', 'Gravida', 'MaternalGlucose', 'AFI',
    # Ultrasound ratios
    'HC_AC_Ratio', 'FL_HC_Ratio', 'FL_AC_Ratio',
    'BPD_AC_Product', 'HadlockScore',
    # Maternal
    'CurrentBMI', 'BMIChange', 'WeightGainPerWeek', 'FH_GA_Ratio',
    # Flags
    'ExcessWeightGain', 'LowWeightGain', 'ObeseMother', 'HighGlucose',
    'Macrosomia_Risk', 'IUGR_Risk', 'Polyhydramnios', 'Oligohydramnios',
]
TARGET_COL = 'ActualBirthWeight_g'

X = df_feat[FEATURE_COLS].values
y = df_feat[TARGET_COL].values

print(f'Feature matrix: {X.shape}  ({len(FEATURE_COLS)} features)')
for i, f in enumerate(FEATURE_COLS, 1):
    print(f'  {i:2d}. {f}')

## 5. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')


# ── Helper evaluation function ────────────────────────────────────────
def evaluate_regression(model, X_te, y_te, name='Model'):
    y_pred = model.predict(X_te)
    rmse   = np.sqrt(mean_squared_error(y_te, y_pred))
    mae    = mean_absolute_error(y_te, y_pred)
    r2     = r2_score(y_te, y_pred)
    # ±10% accuracy: clinically accepted for EFW
    within10 = np.mean(np.abs(y_pred - y_te) / (y_te + 1e-6) <= 0.10) * 100
    within15 = np.mean(np.abs(y_pred - y_te) / (y_te + 1e-6) <= 0.15) * 100
    print(f'{name:<25} RMSE={rmse:.1f}g  MAE={mae:.1f}g  R²={r2:.4f}  ±10%={within10:.1f}%  ±15%={within15:.1f}%')
    return {'Model':name,'RMSE':round(rmse,1),'MAE':round(mae,1),
            'R2':round(r2,4),'±10%':round(within10,1),'±15%':round(within15,1)}

## 6. Baseline: Hadlock Formula vs ML

In [ ]:
# Hadlock baseline
test_df = df_feat.iloc[X_test.shape[0] * -1:].reset_index(drop=True)  # approx slice for display
bpd_te  = X_test[:, FEATURE_COLS.index('BPD')]
hc_te   = X_test[:, FEATURE_COLS.index('HC')]
ac_te   = X_test[:, FEATURE_COLS.index('AC')]
fl_te   = X_test[:, FEATURE_COLS.index('FL')]

hadlock_pred = 10 ** (
    1.3596 + 0.0064*hc_te + 0.0424*ac_te +
    0.174*fl_te + 0.00061*bpd_te*ac_te - 0.00386*ac_te*fl_te
)

rmse_h   = np.sqrt(mean_squared_error(y_test, hadlock_pred))
mae_h    = mean_absolute_error(y_test, hadlock_pred)
r2_h     = r2_score(y_test, hadlock_pred)
w10_h    = np.mean(np.abs(hadlock_pred - y_test) / (y_test + 1e-6) <= 0.10) * 100

print(f'=== BASELINE: Hadlock Formula 4 ===')
print(f'RMSE={rmse_h:.1f}g  MAE={mae_h:.1f}g  R²={r2_h:.4f}  ±10%={w10_h:.1f}%')
print('\nML models should improve over these numbers.')

## 7. Model Comparison

In [ ]:
baseline_regressors = {
    'Random Forest':     RandomForestRegressor(n_estimators=200, random_state=SEED),
    'LightGBM':          lgb.LGBMRegressor(n_estimators=200, random_state=SEED, verbose=-1),
    'XGBoost':           xgb.XGBRegressor(n_estimators=200, random_state=SEED),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=SEED),
    'Ridge Regression':  Ridge(alpha=1.0),
}

results = []
for name, model in baseline_regressors.items():
    model.fit(X_train_sc, y_train)
    r = evaluate_regression(model, X_test_sc, y_test, name)
    results.append(r)

print('\n=== SUMMARY ===')
pd.DataFrame(results).sort_values('RMSE')

## 8. Optuna Tuning (XGBoost Regressor)

In [ ]:
def objective_fbw(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 100, 700),
        'learning_rate':   trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'max_depth':       trial.suggest_int('max_depth', 3, 12),
        'min_child_weight':trial.suggest_int('min_child_weight', 1, 20),
        'subsample':       trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma':           trial.suggest_float('gamma', 0.0, 1.0),
        'reg_alpha':       trial.suggest_float('reg_alpha', 1e-4, 2.0, log=True),
        'reg_lambda':      trial.suggest_float('reg_lambda', 1e-4, 2.0, log=True),
        'random_state': SEED
    }
    model  = xgb.XGBRegressor(**params)
    cv     = KFold(n_splits=5, shuffle=True, random_state=SEED)
    # neg_root_mean_squared_error → we maximize (negate RMSE)
    score  = cross_val_score(model, X_train_sc, y_train,
                              cv=cv, scoring='neg_root_mean_squared_error').mean()
    return score   # already negative; maximize → minimize RMSE

study_fbw = optuna.create_study(direction='maximize')
study_fbw.optimize(objective_fbw, n_trials=100, show_progress_bar=True)

print(f'\nBest CV RMSE: {-study_fbw.best_value:.1f}g')
print(f'Best params: {study_fbw.best_params}')

## 9. Also Tune LightGBM Regressor

In [ ]:
def objective_fbw_lgb(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 700),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 15, 150),
        'max_depth':         trial.suggest_int('max_depth', 3, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 60),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 2.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 2.0, log=True),
        'random_state': SEED, 'verbose': -1
    }
    model = lgb.LGBMRegressor(**params)
    cv    = KFold(n_splits=5, shuffle=True, random_state=SEED)
    score = cross_val_score(model, X_train_sc, y_train,
                             cv=cv, scoring='neg_root_mean_squared_error').mean()
    return score

study_fbw_lgb = optuna.create_study(direction='maximize')
study_fbw_lgb.optimize(objective_fbw_lgb, n_trials=100, show_progress_bar=True)

print(f'\nLightGBM Best CV RMSE: {-study_fbw_lgb.best_value:.1f}g')

## 10. Final Model Selection & Evaluation

In [ ]:
# ── Train both tuned models and pick the better one ──────────────────
params_xgb = study_fbw.best_params
params_xgb['random_state'] = SEED
tuned_xgb = xgb.XGBRegressor(**params_xgb)
tuned_xgb.fit(X_train_sc, y_train)

params_lgb = study_fbw_lgb.best_params
params_lgb.update({'random_state': SEED, 'verbose': -1})
tuned_lgb = lgb.LGBMRegressor(**params_lgb)
tuned_lgb.fit(X_train_sc, y_train)

print('=== TUNED MODEL RESULTS ===')
r_xgb = evaluate_regression(tuned_xgb, X_test_sc, y_test, 'XGBoost (tuned)')
r_lgb = evaluate_regression(tuned_lgb, X_test_sc, y_test, 'LightGBM (tuned)')

# Pick model with lower RMSE
if r_xgb['RMSE'] <= r_lgb['RMSE']:
    fbw_model = tuned_xgb
    print('\n→ Selecting XGBoost as final model')
else:
    fbw_model = tuned_lgb
    print('\n→ Selecting LightGBM as final model')

In [ ]:
# Residual analysis
y_pred_final = fbw_model.predict(X_test_sc)
residuals    = y_test - y_pred_final
pct_error    = (residuals / (y_test + 1e-6)) * 100

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_final, alpha=0.3, s=6, color='#EC4899')
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn, mx], [mn, mx], 'k--', linewidth=1)
axes[0].set_xlabel('Actual Birth Weight (g)')
axes[0].set_ylabel('Predicted Birth Weight (g)')
axes[0].set_title('Actual vs Predicted')

# Residual distribution
axes[1].hist(residuals, bins=40, color='#F9A8D4', edgecolor='white')
axes[1].axvline(0, color='#9D174D', linewidth=2)
axes[1].set_xlabel('Residual (g)')
axes[1].set_title(f'Residual Distribution  (MAE={np.abs(residuals).mean():.0f}g)')

# % error distribution
axes[2].hist(np.abs(pct_error), bins=40, color='#EC4899', edgecolor='white')
axes[2].axvline(10, color='#059669', linewidth=2, label='10% threshold')
axes[2].axvline(15, color='#F59E0B', linewidth=2, label='15% threshold')
axes[2].set_xlabel('Absolute % Error')
axes[2].set_title('Absolute % Error Distribution')
axes[2].legend()

plt.suptitle('Fetal Birth Weight Model — Residual Analysis', fontsize=13, color='#9D174D')
plt.tight_layout()
plt.show()

within10 = np.mean(np.abs(pct_error) <= 10) * 100
within15 = np.mean(np.abs(pct_error) <= 15) * 100
print(f'\n✅ Within ±10%: {within10:.1f}% of predictions (clinical standard: >70%)')
print(f'✅ Within ±15%: {within15:.1f}% of predictions')

## 11. 10-Fold Cross-Validation

In [ ]:
cv10 = KFold(n_splits=10, shuffle=True, random_state=SEED)
cv_rmse = -cross_val_score(fbw_model, X_train_sc, y_train,
                            cv=cv10, scoring='neg_root_mean_squared_error')
print(f'10-Fold CV RMSE: {cv_rmse.mean():.1f}g ± {cv_rmse.std():.1f}g')
print(f'Min: {cv_rmse.min():.1f}g  |  Max: {cv_rmse.max():.1f}g')

## 12. SHAP Feature Importance

In [ ]:
import shap

explainer_fbw = shap.TreeExplainer(fbw_model)
shap_vals_fbw = explainer_fbw.shap_values(X_test_sc)

shap.summary_plot(shap_vals_fbw, X_test_sc, feature_names=FEATURE_COLS,
                  plot_type='bar', show=True)
plt.title('SHAP Feature Importance — Fetal Birth Weight')
plt.show()

In [ ]:
shap.summary_plot(shap_vals_fbw, X_test_sc, feature_names=FEATURE_COLS, show=True)

## 13. Save Model & Scaler

In [ ]:
pickle.dump(fbw_model,    open('model/fbw_model.sav', 'wb'))
pickle.dump(scaler,       open('model/fbw_scaler.sav', 'wb'))
pickle.dump(FEATURE_COLS, open('model/fbw_feature_names.sav', 'wb'))

print('✅ Saved:')
print('   model/fbw_model.sav')
print('   model/fbw_scaler.sav')
print('   model/fbw_feature_names.sav')

## 14. app.py Integration Snippet

In [ ]:
snippet = '''
# ── In app.py load_models() ──────────────────────────────────────────
fbw_model  = pickle.load(open("model/fbw_model.sav",  "rb"))
fbw_scaler = pickle.load(open("model/fbw_scaler.sav", "rb"))

# ── Replace estimate_fetal_birth_weight() with this ──────────────────
def estimate_fetal_birth_weight_ml(gestational_age, bpd, hc, ac, fl,
                                    maternal_weight, weight_gain,
                                    glucose_level, parity, amniotic_fluid,
                                    maternal_age, maternal_height,
                                    pre_pregnancy_bmi, fundal_height, gravida):
    features = engineer_fbw_features(
        gestational_age, maternal_age, maternal_weight, maternal_height,
        pre_pregnancy_bmi, weight_gain, fundal_height, bpd, hc, ac, fl,
        parity, gravida, glucose_level, amniotic_fluid
    )
    features_sc = fbw_scaler.transform(features)
    efw_g       = float(fbw_model.predict(features_sc)[0])
    efw_kg      = efw_g / 1000

    # Re-use the existing classification + signals logic:
    if efw_g > 4500:   category, cat_color = "Severe Macrosomia", "#E11D48"
    elif efw_g > 4000: category, cat_color = "Macrosomia",        "#F59E0B"
    elif efw_g >= 2500: category, cat_color = "Normal Weight",    "#10B981"
    elif efw_g >= 1500: category, cat_color = "Low Birth Weight", "#F59E0B"
    else:               category, cat_color = "Very Low BW",      "#E11D48"

    ga_50th = {36:2600,37:2950,38:3100,39:3250,40:3400,41:3500,42:3550}
    ref     = ga_50th.get(int(gestational_age), 3400)
    pct     = min(max(int(50 * (efw_g / ref)), 3), 97)

    return efw_g, efw_kg, category, cat_color, pct, []
'''
print(snippet)

---
## ✅ All 3 Models Trained!

| Model | File | Task |
|---|---|---|
| GDM Risk | `model/gdm_model.sav` + `model/gdm_scaler.sav` | Binary Classification |
| C-Section | `model/csection_model.sav` + `model/csection_scaler.sav` | Binary Classification |
| Fetal Birth Weight | `model/fbw_model.sav` + `model/fbw_scaler.sav` | Regression |

### Integration Steps
1. Copy all `.sav` files into your project's `model/` folder  
2. In `app.py` `load_models()` — add the 3 new `pickle.load()` lines  
3. Replace the 3 rule-based functions with the ML versions from the snippets above  
4. Run `streamlit run app.py` — done!